# Experiments to show MED = MED+

In [ ]:
import logging
logging.basicConfig(
    filename=f"dev.log",
    filemode="a",
    format="{asctime} {levelname} {filename}:{lineno}: {message}",
    datefmt="%Y-%m-%d %H:%M:%S",
    style="{",
    level=logging.INFO,  # Qiskit dumps too many DEBUG messages
    encoding="utf-8",
)
logging.basicConfig(level=logging.DEBUG)
logging.getLogger('matplotlib.font_manager').disabled = True
logging.getLogger('PIL.PngImagePlugin').disabled = True
logging.getLogger('matplotlib.mathtext').disabled = True
logger = logging.getLogger(__name__)

In [ ]:
import sys
sys.path.append("../")

In [ ]:
import numpy as np
import cvxpy as cp

In [ ]:
from flow.problem_spec import *
from utils.handy_states import *

from flow.solve_mix import *
from utils.inner_product import *
from scipy.spatial.distance import jensenshannon

# Define the input states

In [ ]:
state_dict = sv_simple_2(0.2, 0.5, 0.7)
num_qubits = state_dict["num_qubits"]
num_states = state_dict["num_states"]
state_vec = state_dict["states"]
dense_mat = [DensityMatrix(_) for _ in state_vec]

In [ ]:
"""
num_qubits = 3
num_states = 3
state_vec, dense_mat = sv_coh_asymm_small(num_qubits=num_qubits)
"""

In [6]:
"""
num_qubits = 1
num_states = 2

state_vec = [
    Statevector([1, 0]),
    Statevector([1/np.sqrt(2), 1/np.sqrt(2)]),
]
dense_mat = [DensityMatrix(_) for _ in state_vec]
"""

'\nnum_qubits = 1\nnum_states = 2\n\nstate_vec = [\n    Statevector([1, 0]),\n    Statevector([1/np.sqrt(2), 1/np.sqrt(2)]),\n]\ndense_mat = [DensityMatrix(_) for _ in state_vec]\n'

In [ ]:
"""
num_qubits = 6
num_states = 3

state_vec, _ = sv_coh_asymm_small(num_qubits=num_qubits)
dense_mat = [DensityMatrix(_) for _ in state_vec]
"""

In [8]:
asymm_problem = ProblemSpec(
    num_qubits=num_qubits,
    num_states=num_states,
    case_id="0731_test",
    state_type="densitymatrix",
)

## See how depolarizing noise affects MED and MED+


In [9]:
noise_levels = [0.1 ** (6 - 0.5 * i) for i in range(11)]
disturbance_states = [
    DensityMatrix(
        ProblemSpec.depolarizing_noise_channel(num_qubits=num_qubits)
    )
    for _ in range(num_states)
]

In [10]:
def med_and_med_plus(
    asymm_problem,
    noise_level,
) -> tuple[cp.Problem, cp.Problem]:
    combined_states = [
        (1 - noise_level) * dense_mat[_]
        + noise_level * disturbance_states[_].data
        for _ in range(num_states)
    ]
    asymm_problem.set_states(
        state_type="densitymatrix",
        states=combined_states,
        overwrite=True,
    )
    cvxpy_med_problem = med_problem(asymm_problem)
    cvxpy_med_plus_problem = med_plus_problem(asymm_problem)
    cvxpy_settings = {
        "solver": cp.SCS,
        "verbose": False,
        "requires_grad": True,
        "mkl": True,
        "eps_abs": 1e-6,
        "eps_rel": 1e-5,
        "acceleration_lookback": 10,
        # "warm_start": True,
    }
    cvxpy_settings = {
        "solver": cp.MOSEK,
        "verbose": False,
        "requires_grad": False,
        "eps": 1e-8,
        "mosek_params": {
            "MSK_IPAR_NUM_THREADS": 0,
        }
    }
    solveQSDProblem(
        cvxpy_qsd_problem=cvxpy_med_problem,
        cvxpy_settings=cvxpy_settings,
    )
    solveQSDProblem(
        cvxpy_qsd_problem=cvxpy_med_plus_problem,
        cvxpy_settings=cvxpy_settings,
    )
    # verify_povm_matrix(med_result.values)
    # verify_povm_matrix(med_plus_result.values)
    return cvxpy_med_problem, cvxpy_med_plus_problem

In [ ]:
for noise_level in noise_levels:
    cvxpy_med_problem, cvxpy_med_plus_problem = med_and_med_plus(
        asymm_problem=asymm_problem,
        noise_level=noise_level
    )
    # print(np.round(cvxpy_med_plus_problem.variables()[-1].value, 4))
    print(cvxpy_med_plus_problem.variables()[-1].value)
    print(f"Noise level {np.format_float_scientific(noise_level, precision=4)}")
    print(np.format_float_scientific(cvxpy_med_problem.solution.opt_val, precision=4))
    print(np.format_float_scientific(cvxpy_med_plus_problem.solution.opt_val, precision=4))
    print("\nMED >= MED+?")
    print(cvxpy_med_problem.solution.opt_val >= cvxpy_med_plus_problem.solution.opt_val)
    print(cvxpy_med_problem.solution.opt_val - cvxpy_med_plus_problem.solution.opt_val)


/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18617: UserWarning: Argument sub in putvarboundlist: Incorrect array format causing data to be copied
  warnings.warn("Argument sub in putvarboundlist: Incorrect array format causing data to be copied");
/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18925: UserWarning: Argument subj in putclist: Incorrect array format causing data to be copied
  warnings.warn("Argument subj in putclist: Incorrect array format causing data to be copied");
/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18349: UserWarning: Argument sub in putconboundlist: Incorrect array format causing data to be copied
  warnings.warn("Argument sub in putconboundlist: Incorrect array format causing data to be copied");


[[ 2.70428819e-02+0.00000000e+00j -1.98827031e-02+3.44378519e-02j
  -2.02272670e-02-3.50346543e-02j ...  4.17065118e-18+1.55538068e-18j
  -7.66468667e-18-1.83935349e-17j  3.01669028e-18+1.45385995e-18j]
 [-1.98827031e-02-3.44378519e-02j  6.67309602e-02+0.00000000e+00j
  -3.90190099e-02+6.75829074e-02j ... -3.60729371e-18-1.98480820e-18j
  -4.77127141e-18-5.32159418e-18j  8.41287203e-19-2.45792779e-19j]
 [-2.02272670e-02+3.50346543e-02j -3.90190099e-02-6.75829074e-02j
   1.05767873e-01+0.00000000e+00j ... -7.33976600e-19+1.18536984e-17j
  -2.04591637e-17+2.34363219e-17j -1.16797089e-18-1.10308085e-17j]
 ...
 [ 4.17065118e-18-1.55538068e-18j -3.60729371e-18+1.98480820e-18j
  -7.33976600e-19-1.18536984e-17j ...  2.46742543e-01+0.00000000e+00j
  -5.36341398e-16+2.59821465e-16j -7.57055327e-16-4.27565194e-16j]
 [-7.66468667e-18+1.83935349e-17j -4.77127141e-18+5.32159418e-18j
  -2.04591637e-17-2.34363219e-17j ... -5.36341398e-16-2.59821465e-16j
   2.46742543e-01+0.00000000e+00j -4.69233119e-